# Benchmark: Classical ML Models

**Models** : Linear Regression · SVR · XGBoost · MLP  
**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

## Important

This notebook loads the **frozen canonical splits** produced by  
`01_NiOA_DRNN_Training.ipynb`.  
Do **not** regenerate splits here — identical data must be used for all models.

Set `HORIZON` in Cell 2 to match the horizon whose splits you wish to evaluate.


In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

# Make project importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." , ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.config import SPLITS_DIR, BENCHMARK_DIR, FORECAST_HORIZONS
from benchmarking.utils.data_loader import load_splits, flatten_sequences, combine_train_val
from benchmarking.utils.metrics    import compute_metrics, save_benchmark_results, build_comparison_table

sns.set(style="whitegrid")
print("Environment ready.")

In [ ]:
# ============================================================
#  SET HORIZON before running.  Must match an existing split.
# ============================================================
HORIZON = 60

X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
    load_splits(SPLITS_DIR, HORIZON)

# Flat versions for classical ML
X_tr_flat = flatten_sequences(X_train)
X_va_flat = flatten_sequences(X_val)
X_te_flat = flatten_sequences(X_test)

# Combined train + val for final fitting
X_tv_flat, y_tv = combine_train_val(X_tr_flat, y_train, X_va_flat, y_val)

print(f"Horizon k = {HORIZON} s")
print(f"Flat train+val shape : {X_tv_flat.shape}")
print(f"Test shape           : {X_te_flat.shape}")

---
## 1 · Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_tv_flat, y_tv)
y_pred_lr = lr.predict(X_te_flat)

metrics_lr = save_benchmark_results(
    "linear_regression", HORIZON, y_test, y_pred_lr, BENCHMARK_DIR
)
print("Linear Regression  →", metrics_lr)

---
## 2 · Support Vector Regression

In [ ]:
# SVR can be slow on large flat arrays — consider using a subset if needed.
MAX_SVR = 50_000
X_svr_tr = X_tv_flat[:MAX_SVR]
y_svr_tr = y_tv[:MAX_SVR]

svr = SVR(kernel="rbf", C=10, epsilon=0.1)
svr.fit(X_svr_tr, y_svr_tr)
y_pred_svr = svr.predict(X_te_flat)

metrics_svr = save_benchmark_results(
    "svr", HORIZON, y_test, y_pred_svr, BENCHMARK_DIR,
    extra_meta={"training_subset": MAX_SVR}
)
print("SVR  →", metrics_svr)

---
## 3 · XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators       = 300,
    max_depth          = 6,
    learning_rate      = 0.05,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    early_stopping_rounds = 20,
    eval_metric        = "rmse",
    random_state       = 42,
    verbosity          = 0,
)
xgb.fit(
    X_tr_flat, y_train,
    eval_set   = [(X_va_flat, y_val)],
    verbose    = False,
)
y_pred_xgb = xgb.predict(X_te_flat)

metrics_xgb = save_benchmark_results(
    "xgboost", HORIZON, y_test, y_pred_xgb, BENCHMARK_DIR
)
print("XGBoost  →", metrics_xgb)

---
## 4 · MLP (sklearn)

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes = (128, 64),
    activation         = "relu",
    solver             = "adam",
    learning_rate_init = 5e-4,
    max_iter           = 200,
    early_stopping     = True,
    validation_fraction= 0.1,
    n_iter_no_change   = 15,
    random_state       = 42,
    verbose            = False,
)
mlp.fit(X_tv_flat, y_tv)
y_pred_mlp = mlp.predict(X_te_flat)

metrics_mlp = save_benchmark_results(
    "mlp", HORIZON, y_test, y_pred_mlp, BENCHMARK_DIR
)
print("MLP  →", metrics_mlp)

---
## 5 · Results Summary

In [ ]:
table = build_comparison_table(BENCHMARK_DIR, HORIZON)
print(f"\nBenchmark summary — k = {HORIZON} s")
print(table.to_string(index=False))

# Save aggregated table
out_csv = os.path.join(BENCHMARK_DIR, f"horizon_{HORIZON}", "summary_classical.csv")
table.to_csv(out_csv, index=False)
print(f"\nSummary saved to : {out_csv}")